# Act 1 — Cloud Logging

What used to be called Stackdriver is now a family of products under "Google Cloud Operations." Cloud Logging is where everything starts: it's the central log ingestion, storage, and routing service for every service GCP runs, plus your applications, plus your infrastructure.

Understand the routing model first — it's the part that surprises people coming from other clouds. Every log line is captured in a *project-scoped log bucket*, but you can *also* route copies of selected logs to other destinations (BigQuery, Pub/Sub, GCS, another project's bucket) via **sinks**.

## The Log Router and log buckets

**Log Router** is the engine. Every log line written to a project (by GCP services, by your apps, by GKE, by anything) passes through the Log Router. The Router evaluates **sinks**: each sink has a filter expression and a destination.

**Default sinks per project:**

- `_Required` — non-deletable. Captures Admin Activity and System Event audit logs. 400-day retention, no charge.
- `_Default` — captures everything else. 30-day retention by default. Configurable.

**Log buckets** are the storage destination. Two built-in buckets (`_Required`, `_Default`) plus any you create. Custom buckets can have custom retention (1 day to 10 years) and locations.

**Sinks to other destinations:**

- **BigQuery** — for SQL analysis of logs over time. The headline pattern: stream all org-level audit logs to a central BigQuery dataset for security analysis.
- **Pub/Sub** — for real-time downstream processing (SIEM ingestion, alerting pipelines).
- **Cloud Storage** — for long-term cold storage.
- **Another project's log bucket** — for org-wide centralisation.

**Log Analytics** is a SQL interface over log buckets. Same data, different query path; no separate ETL. Useful for ad-hoc investigation when you don't already have a BigQuery sink set up.

## Patterns and features

- **Exclusion filters** — stop logs you don't need (verbose health checks, debug-level prod logs) from hitting `_Default`. Cheapest way to control logging costs.
- **Log-based metrics** — define a metric (counter or distribution) over a log filter. The metric appears in Cloud Monitoring. Useful when an app emits structured logs but no native metrics.
- **Log scopes** — query across multiple projects from one place (`organizations/N/logs` or aggregated views).
- **Structured logging** — write JSON to stdout from Cloud Run / GKE / GCE; Cloud Logging parses the JSON into searchable fields. Always preferred over plain-text logs.

**Retention rule of thumb:** `_Default` 30 days is fine for most apps; raise to 90 or longer for anything regulated. Audit Logs go to `_Required` for 400 days automatically — don't lower this.

# Act 2 — Cloud Monitoring

Logs are events; metrics are time-series numbers. Cloud Monitoring is the metric, dashboard, and alerting product. Where Cloud Logging asks "what happened?", Cloud Monitoring asks "how is the system performing?"

## Metrics, dashboards, alerting

**Metrics scope** (replacing the older "workspace") defines which projects' metrics a Monitoring view sees. Common shape: one Monitoring scope per environment (prod, dev) that aggregates all projects in that env.

**Sources of metrics:**

- **Built-in metrics** — GCP emits thousands. CPU usage, Cloud Run request counts, Cloud SQL connections, BigQuery slot usage, GCS request rate. No configuration.
- **Custom metrics** — your apps emit via the Monitoring API or OpenTelemetry. Counter, gauge, distribution.
- **Log-based metrics** — derived from log filters (Act 1).
- **Prometheus metrics** — Managed Service for Prometheus ingests Prometheus-format metrics from GKE workloads at scale; queryable via PromQL.

**Query languages:** **MQL** (Monitoring Query Language) for native metrics; **PromQL** for Prometheus metrics. Dashboards mix both.

**Alerting policies** fire on a condition (metric threshold, log match, uptime check failure). Targets are **notification channels** — email, SMS, Slack, PagerDuty, webhooks, Pub/Sub.

**Uptime checks** are synthetic probes from Google's network. Hit a URL every minute from N regions; alert on failure or slow response. The poor-man's external monitoring; useful for catching issues your internal metrics wouldn't see.

## SLOs and burn-rate alerts

**Service-Level Objectives (SLOs)** are Cloud Monitoring's first-class support for SRE-style reliability work.

- **SLI** — Service Level Indicator. A measurable success rate ("% of requests returning 200 within 500ms").
- **SLO** — the target ("99.5% over a rolling 28-day window").
- **Error budget** — the gap between 100% and the SLO. 0.5% of requests can fail without missing the target.
- **Burn-rate alerts** — alert when you're burning the error budget faster than allowed. Two-window/two-threshold patterns ("fast burn" = high burn for short window, "slow burn" = moderate burn for long window) are the canonical SRE approach.

SLOs avoid the "CPU > 80%" page-everyone trap by alerting on *user-visible reliability*, not on a proxy metric that may or may not correlate.

# Act 3 — Trace, Profiler, Error Reporting

Three smaller observability products complete the picture. They're often turned on once and forgotten about until you need them.

## Cloud Trace, Cloud Profiler, Error Reporting

**Cloud Trace** is distributed tracing. Apps emit spans via OpenTelemetry; Cloud Trace stores and visualises them. Cloud Run and App Engine auto-instrument; on GKE / GCE you wire up the OTel SDK. Use to diagnose "which leg of this multi-service call is slow."

**Cloud Profiler** is continuous statistical CPU/memory profiling. The agent runs in your process, samples profile data periodically, and uploads to Profiler. The UI shows flame graphs over time. Used to find regressions ("this version uses 30% more CPU on the same load") and hot paths to optimise.

**Error Reporting** aggregates exceptions across services. App logs an unhandled exception → Error Reporting groups identical stack traces → you see counts and trends per error. Notifications when new error types appear. Useful for the "how often does this exception happen, and when did it start?" question.

# Act 4 — Audit Logs, Asset Inventory, Service Health

The governance side of the chapter. These products answer "what happened in our org?", "what exists in our org?", and "is Google having a problem?"

## Cloud Audit Logs

Every GCP API call generates an audit log. Four streams:

- **Admin Activity** — any API call that modifies config (creating a VM, granting IAM, changing a bucket). **Always on, can't be turned off, free.**
- **Data Access** — reads or writes of data within a service (`storage.objects.get`, `bigquery.queries.create`). **Opt-in per service**, can generate huge volumes. Critical for sensitive workloads.
- **System Event** — Google-initiated actions (live migration of a VM, automatic maintenance). Always on, free.
- **Policy Denied** — any API call that was rejected by IAM, VPC-SC, or Org Policy. Always on, free.

**Routing for analysis.** The standard pattern: aggregate sink at the Org level → BigQuery dataset in a central project → scheduled queries answering "who created an SA key in the last 24h?" "which projects had Policy Denied events?" "which service accounts impersonated other SAs?"

**Combine with Security Command Center** (notebook 11) for ML-driven anomaly detection on top of the same log stream.

## Cloud Asset Inventory

**Cloud Asset Inventory** is a queryable index of every resource and IAM binding in your Org. Three primary uses:

- **Search** — "every VM in europe-west1 with a public IP" or "every IAM binding granting `roles/owner` at folder X."
- **Export** — full snapshot of asset state to BigQuery or GCS, point-in-time. Used for compliance reports.
- **Change feed** — Pub/Sub notifications on asset changes. Used for real-time policy enforcement ("if a public bucket is created, alert security").

It's an underrated tool. Most teams use it for the "do I have any X anywhere?" question that would otherwise require iterating every project.

## Personalised Service Health and the public status dashboard

**Personalised Service Health** is a per-project surface of incidents Google has detected affecting *your* projects (not the whole platform). Routed to Pub/Sub for downstream automation.

The **public status dashboard** at `status.cloud.google.com` shows ongoing platform incidents. Worth bookmarking for "is it just us?" questions during incidents.

## What carries into later chapters

Observability is a substrate the whole course sits on. Cloud Run logs and metrics (notebook 04) all flow to Logging and Monitoring. SLO burn-rate alerts gate canary deploys (notebook 13). Audit Logs feed Security Command Center findings (notebook 11). BigQuery exports of audit logs and Asset Inventory snapshots are the substrate for compliance reporting (notebook 14).

Three habits to carry forward:

- **Structured JSON logs from every app.** Plain-text logs cripple your future debugging.
- **SLOs before page-on-CPU thresholds.** Alert on user-visible reliability, not on proxies.
- **Aggregated Audit Log sink at the Org level into BigQuery.** Set this up once; the value compounds forever.